In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import json

with open('/content/drive/MyDrive/mit-bih/beat_symbols.json', encoding='utf-8') as file:
    beat_syms = json.load(file)

In [8]:
import pandas as pd

for k in beat_syms.keys():
    print(f'k: {k}')
    print(pd.Series(beat_syms[k]).value_counts(), end='\n\n')

k: 100
N    2239
S      33
Q       1
V       1
Name: count, dtype: int64

k: 101
N    1860
Q      11
S       3
Name: count, dtype: int64

k: 103
N    2082
Q       7
S       2
Name: count, dtype: int64

k: 105
N    2526
Q     124
V      41
Name: count, dtype: int64

k: 106
N    1507
V     520
Q      71
Name: count, dtype: int64

k: 107
Q    2081
V      59
Name: count, dtype: int64

k: 108
N    1740
Q      61
V      17
S       4
F       2
Name: count, dtype: int64

k: 109
N    2492
V      38
Q       3
F       2
Name: count, dtype: int64

k: 111
N    2123
Q       9
V       1
Name: count, dtype: int64

k: 112
N    2537
Q      11
S       2
Name: count, dtype: int64

k: 113
N    1789
S       6
Q       1
Name: count, dtype: int64

k: 114
N    1820
V      43
S      12
Q      11
F       4
Name: count, dtype: int64

k: 115
N    1953
Q       9
Name: count, dtype: int64

k: 116
N    2302
V     109
Q       9
S       1
Name: count, dtype: int64

k: 117
N    1534
Q       4
S       1
Name: count, dtyp

In [9]:
import torch
import torch.nn as nn


def beat_to_idx(symbols: list[str]) -> list[int]:
    aami_mapping = {
        # class N
        'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
        # class S
        'S': 1, 'A': 1, 'a': 1, 'J': 1,
        # class V
        'V': 2, 'E': 2,
        # class F
        'F': 3
    }
    return [aami_mapping.get(sym, 4) for sym in symbols]


def calc_weights(seq: list[int]):
    N = len(seq)
    counts = pd.Series(seq).value_counts()
    K = len(counts)
    weights = N / (K * counts)
    weights = torch.tensor([weights.get(i, 0) for i in range(CharTransformer.V)], dtype=torch.float)
    return weights / weights.mean()


class CharTransformer(nn.Module):
    SEQ_LEN: int = 200  # context window
    H: int = 20  # horizon
    V: int = 5  # vocab size

    def __init__(
            self,
            d_model: int = 64,
            nhead: int = 4,
            num_layers: int = 2
    ):
        super().__init__()

        self.embed = nn.Embedding(self.V, d_model)
        self.pos = nn.Embedding(self.SEQ_LEN, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=128,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)

        self.head = nn.Linear(d_model, self.H * self.V)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        out = self.embed(x) + self.pos(pos)
        out = self.transformer(out)

        last = out[:, -1, :]
        logits = self.head(last)
        logits = logits.view(B, self.H, -1)
        return logits

In [10]:
data_id = '208'
seq = beat_to_idx(beat_syms[data_id])
print(pd.Series(seq).value_counts(), end='\n\n')

0    1586
2     992
3     373
4      87
1       2
Name: count, dtype: int64



In [11]:
SEQ_LEN = CharTransformer.SEQ_LEN
H = CharTransformer.H
V = CharTransformer.V

In [12]:
train_seq = seq[:-H]
test_seq = seq[-H:]

In [13]:
X, y = [], []
N = len(train_seq)
for i in range(N - SEQ_LEN - H + 1):
    X.append(train_seq[i: i + SEQ_LEN])
    y.append(train_seq[i + SEQ_LEN: i + SEQ_LEN + H])

In [14]:
X = torch.tensor(X)
y = torch.tensor(y)

In [15]:
print(f"Training samples: {len(X)} | context={SEQ_LEN} | horizon H={H}")

Training samples: 2801 | context=200 | horizon H=20


In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CharTransformer().to(device)

In [17]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y)
loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True if device.type == 'cuda' else False,
    num_workers=2
    )

In [18]:
from torch.optim.lr_scheduler import OneCycleLR
from torch.nn.utils import clip_grad_norm_

EPOCHS = 100
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = OneCycleLR(
    opt,
    max_lr=3e-3,
    epochs=EPOCHS,
    steps_per_epoch=len(loader)
)

weights = calc_weights(train_seq).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)

        logits = model(batch_X)

        loss = loss_fn(logits.reshape(-1, V), batch_y.reshape(-1))

        opt.zero_grad()
        loss.backward()

        clip_grad_norm_(model.parameters(), max_norm=1.0)

        opt.step()
        scheduler.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}, Loss: {epoch_loss / len(loader):.4f}")

Epoch 1, Loss: 1.6906
Epoch 2, Loss: 1.6214
Epoch 3, Loss: 1.5931
Epoch 4, Loss: 1.5844
Epoch 5, Loss: 1.5536
Epoch 6, Loss: 1.5431
Epoch 7, Loss: 1.5067
Epoch 8, Loss: 1.4904
Epoch 9, Loss: 1.4419
Epoch 10, Loss: 1.3984
Epoch 11, Loss: 1.3805
Epoch 12, Loss: 1.3430
Epoch 13, Loss: 1.3487
Epoch 14, Loss: 1.2907
Epoch 15, Loss: 1.2475
Epoch 16, Loss: 1.2065
Epoch 17, Loss: 1.1439
Epoch 18, Loss: 1.1210
Epoch 19, Loss: 1.1207
Epoch 20, Loss: 1.0888
Epoch 21, Loss: 1.0338
Epoch 22, Loss: 1.0497
Epoch 23, Loss: 1.0278
Epoch 24, Loss: 1.0571
Epoch 25, Loss: 1.0105
Epoch 26, Loss: 0.9746
Epoch 27, Loss: 0.9731
Epoch 28, Loss: 0.9560
Epoch 29, Loss: 0.9725
Epoch 30, Loss: 0.9722
Epoch 31, Loss: 0.9260
Epoch 32, Loss: 0.9647
Epoch 33, Loss: 0.9098
Epoch 34, Loss: 0.9311
Epoch 35, Loss: 0.9075
Epoch 36, Loss: 0.9030
Epoch 37, Loss: 0.8790
Epoch 38, Loss: 0.8617
Epoch 39, Loss: 0.8642
Epoch 40, Loss: 0.8445
Epoch 41, Loss: 0.8464
Epoch 42, Loss: 0.8220
Epoch 43, Loss: 0.8140
Epoch 44, Loss: 0.79

In [19]:
model.eval()

CharTransformer(
  (embed): Embedding(5, 64)
  (pos): Embedding(200, 64)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Linear(in_features=64, out_features=100, bias=True)
)

In [20]:
context = torch.tensor([train_seq[-SEQ_LEN:]], device=device)
with torch.no_grad():
    logits = model(context)
    pred_idx = logits.argmax(dim=-1).squeeze(0)

In [21]:
print("Forecast:")
print(pred_idx)
print("Actual:")
test_seq = torch.tensor(test_seq, device=device)
print(test_seq)

Forecast:
tensor([2, 4, 0, 2, 2, 0, 2, 2, 0, 0, 2, 0, 0, 3, 2, 0, 2, 2, 0, 2],
       device='cuda:0')
Actual:
tensor([2, 0, 0, 2, 0, 0, 2, 4, 0, 3, 2, 0, 2, 0, 3, 2, 0, 2, 0, 0],
       device='cuda:0')


In [22]:
torch.save(model.state_dict(), 'hb_forecaster_model.pth')